# Bag-of-Words
* CountVectorizer + Reg Softmax
* TF-IDF + Reg Softmax

In [1]:
import torch
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from torch.utils.data import Dataset, DataLoader, random_split

In [8]:
class FlipkartDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
        
    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [9]:
metadata = pd.read_csv('../datasets/Flipkart/flipkart_com-ecommerce_sample_1050.csv')
count_vectorizer = CountVectorizer()
x_countvec = torch.tensor(count_vectorizer.fit_transform(metadata['description'].values).todense(), dtype=torch.float32)

tfidf_vectorizer = TfidfVectorizer(stop_words='english', max_features=500)
x_tfidfvec = torch.tensor(tfidf_vectorizer.fit_transform(metadata['description'].values).todense(), dtype=torch.float32)

print(x_countvec.shape)
print(x_tfidfvec.shape)

print(count_vectorizer.vocabulary_)
print(tfidf_vectorizer.vocabulary_)

torch.Size([1050, 6053])
torch.Size([1050, 500])
{'key': 3289, 'features': 2488, 'of': 3984, 'elegance': 2277, 'polyester': 4328, 'multicolor': 3809, 'abstract': 790, 'eyelet': 2431, 'door': 2152, 'curtain': 1928, 'floral': 2581, '213': 239, 'cm': 1656, 'in': 3088, 'height': 2935, 'pack': 4097, 'price': 4406, 'rs': 4785, '899': 718, 'this': 5519, 'enhances': 2315, 'the': 5497, 'look': 3490, 'interiors': 3171, 'is': 3194, 'made': 3554, 'from': 2657, '100': 37, 'high': 2960, 'quality': 4502, 'fabric': 2437, 'it': 3201, 'an': 933, 'style': 5315, 'stitch': 5261, 'with': 5958, 'metal': 3685, 'ring': 4721, 'makes': 3574, 'room': 4757, 'environment': 2337, 'romantic': 4753, 'and': 939, 'loving': 3517, 'ant': 958, 'wrinkle': 6004, 'anti': 962, 'shrinkage': 5011, 'have': 2909, 'elegant': 2278, 'apparance': 980, 'give': 2757, 'your': 6033, 'home': 2992, 'bright': 1367, 'modernistic': 3743, 'appeal': 983, 'these': 5507, 'designs': 2045, 'surreal': 5385, 'attention': 1068, 'sure': 5375, 'to': 5566

In [10]:
le = LabelEncoder()
y = metadata['product_category_tree'].apply(lambda x: x.split(' >> ')[0][2:])
y = le.fit_transform(y)

y = torch.tensor(y, dtype=torch.long)

In [11]:
import torch
import torch.optim as optim
from sklearn.metrics import accuracy_score
import numpy as np
from torch import nn

# --- Hyperparameters ---
BATCH_SIZE = 2
LR = 1e-4
n_epochs = 50
PATIENCE = 5  # early stopping patience
device = 'cuda' if torch.cuda.is_available() else 'cpu'  # Très lent en CPU

# --- Dataset / DataLoader ---
dataset = FlipkartDataset(x_countvec, y)

val_ratio = 0.2
val_size = int(len(dataset) * val_ratio)
train_size = len(dataset) - val_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

num_classes = 7

def compute_mlp(X):
    eta = 1e-1
    hidden_dim = 64  # Dimension de la couche cachée du MLP
    
    # --- Définition de l'architecture MLP ---
    class MLP(nn.Module):
        def __init__(self, input_dim, hidden_dim, output_dim):
            super(MLP, self).__init__()
            self.network = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, output_dim)
            )
            
        def forward(self, t):
            return self.network(t)

    # Instanciation du modèle, de la fonction de coût et de l'optimiseur
    model = MLP(X.shape[1], hidden_dim, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()  # Inclut déjà le Softmax numériquement stable
    optimizer = optim.SGD(model.parameters(), lr=eta)

    # --- Boucle d'entraînement ---
    for epoch in range(n_epochs):
        for batch_x, batch_y in train_loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            # Forward pass
            outputs = model(batch_x)
            
            # CrossEntropyLoss de PyTorch attend des indices de classe (LongTensor) 
            # ou des probabilités cibles. On s'adapte à votre 'y' au format One-Hot:
            targets = torch.argmax(batch_y, dim=-1)
            loss = criterion(outputs, targets)
            
            # Backward pass et mise à jour des poids
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # --- Métriques (Évaluation à la fin de chaque époque) ---
        with torch.no_grad():
            full_outputs = model(X)
            preds = torch.argmax(full_outputs, dim=-1)
            targets_all = torch.argmax(y, dim=-1)
            accuracy = (preds == targets_all).float().mean().item() * 100
            print(round(accuracy, 2))


In [12]:
compute_mlp(x_countvec)

ValueError: Expected input batch_size (2) to match target batch_size (0).

In [ ]:
x_countvec.shape

In [ ]:
x_countvec.amax(axis=1).shape

In [ ]:
# m = x_countvec.shape[0]

# x_countvec /= x_countvec.amax(axis=1, keepdim=True)
# x_tfidfvec /= x_tfidfvec.amax(axis=1, keepdim=True)


# x_countvec = torch.hstack([x_countvec, torch.ones((m, 1))])
# x_tfidfvec = torch.hstack([x_tfidfvec, torch.ones((m, 1))])



In [ ]:
compute_mlp(x_countvec)

In [ ]:
compute_mlp(x_tfidfvec)

# BERT + MLP sur Flipkart
Voir https://sbert.net/ 

Pour la liste des modèles préentrainés : https://sbert.net/docs/sentence_transformer/pretrained_models.html 

In [18]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(metadata['description'].values.tolist())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
embeddings.shape

(1050, 768)

In [22]:
compute_mlp(torch.tensor(embeddings, dtype=torch.float))

RuntimeError: mat1 and mat2 shapes cannot be multiplied (2x6053 and 768x64)

# Full Transformers sur Flipkart

In [23]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re

# ===========================
# Parameters
# ===========================
MAX_SEQ_LEN = 16
EMBED_DIM = 128
NUM_HEADS = 2
FF_DIM = 32
NUM_LAYERS = 2
DROPOUT = 0.2
BATCH_SIZE = 8
NUM_CLASSES = 7
LR = 1e-3
EPOCHS = 30
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

texts = metadata['description'].values.tolist()

y_tensor = torch.argmax(y, dim=1)

# ===========================
# Tokenizer + vocab
# ===========================
def simple_tokenizer(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text.split()

all_tokens = [token for text in texts for token in simple_tokenizer(text)]
vocab = {token: i+1 for i, (token, _) in enumerate(Counter(all_tokens).most_common())}
vocab_size = len(vocab) + 1

def encode(text):
    tokens = simple_tokenizer(text)
    ids = [vocab.get(tok,0) for tok in tokens]
    if len(ids) < MAX_SEQ_LEN:
        ids += [0]*(MAX_SEQ_LEN - len(ids))
    else:
        ids = ids[:MAX_SEQ_LEN]
    return ids

X_tensor = torch.tensor([encode(t) for t in texts], dtype=torch.long)

# ===========================
# Dataset & DataLoader
# ===========================
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = TextDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

# ===========================
# Positional Encoding
# ===========================
class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, max_len=MAX_SEQ_LEN):
        super().__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-torch.log(torch.tensor(10000.0)) / embed_dim))
        pe[:,0::2] = torch.sin(position * div_term)
        pe[:,1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(1))  # (max_len, 1, embed_dim)

    def forward(self, x):
        # x: (seq_len, batch, embed_dim)
        x = x + self.pe[:x.size(0), :]
        return x

# ===========================
# Transformer Encoder Model using nn.TransformerEncoder
# ===========================
class TransformerClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim, num_layers, num_classes, max_len=MAX_SEQ_LEN, dropout=DROPOUT):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.pos_embed = PositionalEncoding(embed_dim, max_len)

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embed(x)                  # (batch, seq_len, embed_dim)
        x = x.transpose(0,1)               # (seq_len, batch, embed_dim)
        x = self.pos_embed(x)              # add positional encoding
        x = self.transformer_encoder(x)    # (seq_len, batch, embed_dim)
        x = x.mean(dim=0)                  # global average pooling over sequence
        out = self.classifier(x)
        return out

# ===========================
# Instantiate model
# ===========================
model = TransformerClassifier(vocab_size=vocab_size,
                              embed_dim=EMBED_DIM,
                              num_heads=NUM_HEADS,
                              ff_dim=FF_DIM,
                              num_layers=NUM_LAYERS,
                              num_classes=NUM_CLASSES).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.CrossEntropyLoss()

from sklearn.model_selection import train_test_split

# ===========================
# Split train/validation
# ===========================
X_train, X_val, y_train, y_val = train_test_split(
    X_tensor, y_tensor, test_size=0.2, random_state=42, stratify=y_tensor
)

train_dataset = TextDataset(X_train, y_train)
val_dataset = TextDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

# ===========================
# Training loop with validation accuracy
# ===========================
for epoch in range(EPOCHS):
    # --- training ---
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
        preds = torch.argmax(outputs, dim=1)
        correct += (preds == y_batch).sum().item()
        total += y_batch.size(0)
    
    train_loss = total_loss / len(train_loader)
    train_acc = correct / total * 100

    # --- validation ---
    model.eval()
    val_correct = 0
    val_total = 0
    val_loss_total = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            val_loss_total += loss.item()
            
            preds = torch.argmax(outputs, dim=1)
            val_correct += (preds == y_batch).sum().item()
            val_total += y_batch.size(0)
    
    val_loss = val_loss_total / len(val_loader)
    val_acc = val_correct / val_total * 100

    print(f"Epoch {epoch+1}/{EPOCHS}, "
          f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%")


IndexError: Dimension out of range (expected to be in range of [-1, 0], but got 1)

In [ ]:
# ===========================
# Transformer Encoder Model using nn.TransformerEncoder
# ===========================
class TransformerClassifier(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, num_layers, num_classes, max_len=MAX_SEQ_LEN, dropout=DROPOUT):
        super().__init__()

        self.pos_embed = PositionalEncoding(embed_dim, max_len)

        encoder_layer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout)
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        
        self.classifier = nn.Linear(embed_dim, 1)

    def forward(self, x):
        # x: (batch, seq_len)
        x = self.embed(x)                  # (batch, seq_len, embed_dim)
        x = x.transpose(0,1)               # (seq_len, batch, embed_dim)
        x = self.pos_embed(x)              # add positional encoding
        x = self.transformer_encoder(x)    # (seq_len, batch, embed_dim)
        x = x.mean(dim=0)                  # global average pooling over sequence
        out = self.classifier(x)
        return out

# CNN vs ViT

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time

# ===========================
# Hyperparameters
# ===========================
batch_size = 128
num_classes = 10
num_epochs = 3
learning_rate = 0.001
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===========================
# Dataset
# ===========================
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# ===========================
# LeNet5_GAP definition
# ===========================
class LeNet5_GAP(nn.Module):
    def __init__(self, num_classes=10):
        super(LeNet5_GAP, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, kernel_size=5)  # 3 channels for CIFAR10
        self.pool1 = nn.AvgPool2d(2)
        self.conv2 = nn.Conv2d(6, 16, kernel_size=5)
        self.pool2 = nn.AvgPool2d(2)
        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(16, num_classes)

    def forward(self, x):
        x = torch.tanh(self.conv1(x))
        x = self.pool1(x)
        x = torch.tanh(self.conv2(x))
        x = self.pool2(x)
        x = self.gap(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# ===========================
# ViT definition (8x8 patches)
# ===========================
import torch
import torch.nn as nn
import torch.nn.functional as F

# ===========================
# Transformer Encoder Block
# ===========================
class TransformerEncoderBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_dim, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim),
            nn.Dropout(dropout)
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (seq_len, batch, embed_dim)
        x_norm = self.norm1(x)
        attn_out, _ = self.attn(x_norm, x_norm, x_norm)
        x = x + self.dropout(attn_out)
        x_norm = self.norm2(x)
        x = x + self.mlp(x_norm)
        return x

# ===========================
# ViT Model
# ===========================
class ViT(nn.Module):
    def __init__(self, image_size=32, patch_size=8, in_channels=3, num_classes=10,
                 embed_dim=128, depth=6, num_heads=4, mlp_dim=256, dropout=0.1):
        super().__init__()
        assert image_size % patch_size == 0, "Image size must be divisible by patch size"
        self.num_patches = (image_size // patch_size) ** 2
        self.patch_size = patch_size
        self.embed_dim = embed_dim

        # Patch embedding
        self.patch_embed = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

        # CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, embed_dim))
        self.dropout = nn.Dropout(dropout)

        # Transformer encoder
        self.blocks = nn.ModuleList([
            TransformerEncoderBlock(embed_dim, num_heads, mlp_dim, dropout) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)

        # Classifier head
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialize weights
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)

    def forward(self, x):
        # x: (B, C, H, W)
        B = x.shape[0]
        x = self.patch_embed(x)  # (B, embed_dim, H/patch, W/patch)
        x = x.flatten(2).transpose(1, 2)  # (B, num_patches, embed_dim)

        cls_tokens = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat((cls_tokens, x), dim=1)          # (B, num_patches+1, embed_dim)
        x = x + self.pos_embed
        x = self.dropout(x)

        # Transformer blocks (need seq_len, batch, embed_dim for nn.MultiheadAttention)
        x = x.transpose(0, 1)  # (seq_len, batch, embed_dim)
        for block in self.blocks:
            x = block(x)
        x = x.transpose(0, 1)  # (B, seq_len, embed_dim)

        x = self.norm(x[:, 0])  # Take CLS token
        x = self.head(x)
        return x

vit_model = ViT(image_size=32, patch_size=8, in_channels=3, num_classes=10)


# ===========================
# Training and evaluation functions
# ===========================
def train_model(model, criterion, optimizer, num_epochs):
    model = model.to(device)
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        start_time = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_loss = running_loss / len(train_loader.dataset)
        epoch_acc = correct / total
        print(f"Epoch [{epoch+1}/{num_epochs}] - Loss: {epoch_loss:.4f} - Acc: {epoch_acc:.4f} - Time: {time.time()-start_time:.1f}s")

def evaluate_model(model):
    model.eval()
    model = model.to(device)
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    acc = correct / total
    print(f"Test Accuracy: {acc:.4f}")
    return acc

# ===========================
# Train and evaluate LeNet5_GAP
# ===========================
print("=== Training LeNet5_GAP ===")
lenet_model = LeNet5_GAP(num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(lenet_model.parameters(), lr=learning_rate)

train_model(lenet_model, criterion, optimizer, num_epochs)
acc_lenet = evaluate_model(lenet_model)

# ===========================
# Train and evaluate ViT
# ===========================
print("\n=== Training ViT (8x8 patches) ===")
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(vit_model.parameters(), lr=learning_rate)

train_model(vit_model, criterion, optimizer, num_epochs)
acc_vit = evaluate_model(vit_model)

print(f"\nLeNet5_GAP Accuracy: {acc_lenet:.4f}")
print(f"ViT Accuracy: {acc_vit:.4f}")
